# Memory-Native Qwen3.5-27B — Strict KD v2

**Target:** one NVIDIA H100 (80 GB) or RTX PRO 6000 Blackwell (96 GB).  
This notebook protects the good warm conversion and accepts KD only after strict
`alpha=0` aggregate and per-domain gates. Run every cell in order.


In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=False)


In [ ]:
# ---- User paths ----
from pathlib import Path
import shutil

RELEASE_ZIP = Path('/content/drive/MyDrive/mn_strict_kd_qwen35_27b_h100_rtxpro6000_v2.zip')
DONOR_DIR   = Path('/content/drive/MyDrive/colab_models/Qwen3.8-27B')
WARM_STATE  = Path('/content/drive/MyDrive/mn27/state_max.zip')
OUTPUT_DRIVE = Path('/content/drive/MyDrive/mn27_strict_v2')

WORK = Path('/content/mn_strict_v2')
PROJECT = WORK / 'project'
LOCAL_OUTPUT = WORK / 'output'
if LOCAL_OUTPUT.exists(): shutil.rmtree(LOCAL_OUTPUT)
for p in (WORK, LOCAL_OUTPUT): p.mkdir(parents=True, exist_ok=True)
OUTPUT_DRIVE.parent.mkdir(parents=True, exist_ok=True)
assert RELEASE_ZIP.exists(), RELEASE_ZIP
assert DONOR_DIR.exists(), DONOR_DIR
assert WARM_STATE.exists(), WARM_STATE


In [ ]:
# Hardware, disk, and CUDA preflight.
import os, shutil, subprocess, sys, json, torch
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True,max_split_size_mb:256'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
assert torch.cuda.is_available(), 'CUDA GPU required'
props = torch.cuda.get_device_properties(0)
name, gib = props.name, props.total_memory / 2**30
print(f'GPU: {name} | {gib:.1f} GiB | bf16={torch.cuda.is_bf16_supported()}')
assert gib >= 72, 'This 27B recipe requires at least 72 GiB VRAM'
assert torch.cuda.is_bf16_supported(), 'BF16 support required'
free = shutil.disk_usage('/content').free / 2**30
print(f'Local free disk: {free:.1f} GiB')
assert free >= 35, 'Need >=35 GiB local free disk; the notebook avoids duplicating the donor'


In [ ]:
# Extract release and install helpers without replacing Colab's CUDA-matched torch.
import shutil, zipfile, subprocess, sys
if PROJECT.exists(): shutil.rmtree(PROJECT)
tmp = WORK / 'extract_tmp'
if tmp.exists(): shutil.rmtree(tmp)
tmp.mkdir(parents=True)
with zipfile.ZipFile(RELEASE_ZIP) as zf: zf.extractall(tmp)
roots = [p for p in tmp.iterdir() if p.is_dir()]
src = roots[0] if len(roots) == 1 else tmp
shutil.move(str(src), str(PROJECT))
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(PROJECT), '--no-deps'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(PROJECT/'requirements-colab.txt')], check=True)


In [ ]:
# Ensure this Transformers build knows Qwen3.5. Install from upstream only when necessary.
import importlib, subprocess, sys
ok = False
try:
    import transformers
    ok = hasattr(transformers, 'Qwen3_5ForConditionalGeneration') or 'qwen3_5' in str(getattr(transformers.models, '__all__', '')).lower()
except Exception:
    pass
if not ok:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-U', 'git+https://github.com/huggingface/transformers.git'], check=True)
    importlib.invalidate_caches()
print('Transformers compatibility check complete')


In [ ]:
# Validate donor shards, architecture, warm zip, patched runner, and GPU.
import subprocess, sys
preflight = LOCAL_OUTPUT / 'preflight.json'
subprocess.run([
    sys.executable, str(PROJECT/'scripts/validate_strict_release.py'),
    '--donor', str(DONOR_DIR), '--state', str(WARM_STATE),
    '--output', str(preflight), '--require-gpu'
], check=True)
print(preflight.read_text())


## Storage policy

The donor remains on Drive unless at least 90 GiB of local disk is free. The warm
state is extracted into `/dev/shm` when RAM-backed storage has enough room. This
avoids the earlier 91–95% disk saturation and duplicate 50+ GiB model copy.


In [ ]:
import shutil, os, subprocess, zipfile
local_free = shutil.disk_usage('/content').free / 2**30
if local_free >= 90:
    LOCAL_DONOR = WORK/'donor'
    if not (LOCAL_DONOR/'model.safetensors.index.json').exists():
        subprocess.run(['rsync','-a','--info=progress2',str(DONOR_DIR)+'/',str(LOCAL_DONOR)+'/'], check=True)
else:
    LOCAL_DONOR = DONOR_DIR

shm_free = shutil.disk_usage('/dev/shm').free / 2**30
STATE_DIR = Path('/dev/shm/mn27_state') if shm_free >= 26 else WORK/'mn27_state'
if not STATE_DIR.exists() or not any(STATE_DIR.iterdir()):
    STATE_DIR.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(WARM_STATE) as zf: zf.extractall(STATE_DIR)
print('DONOR=', LOCAL_DONOR, 'STATE=', STATE_DIR)


## Teacher cache

Use the bundled cache builder, but create a **v2 cache** containing full-vocabulary
teacher `logsumexp` at T=1. A cache with only renormalized top-k logits must not be
used. The helper below locates the bundled cache script and prints its CLI before
execution; edit only the argument mapping cell if the legacy script exposes a
different spelling.


In [ ]:
# Audit the exact bundled runner before any expensive work.
import pathlib, subprocess, sys, json
audit_path = LOCAL_OUTPUT/'legacy_runner_audit.json'
subprocess.run([
    sys.executable, str(PROJECT/'scripts/audit_legacy_runner.py'),
    str(PROJECT/'scripts/kd_cached_recovery.py'), '--output', str(audit_path)
], check=True)
audit = json.loads(audit_path.read_text())
print(json.dumps(audit, indent=2, ensure_ascii=False))
assert audit['strict_patch_marker']


In [ ]:
# Runtime recipe. H100 and RTX PRO 6000 use the same correctness settings;
# only cache/build throughput may differ.
import json, torch
from mn_strict_kd.hardware import detect, as_env
profile = detect()
recipe = json.loads((PROJECT/'configs/qwen35_27b_strict_kd_v2.json').read_text())
recipe['hardware'] = profile.__dict__
recipe['paths'] = {
    'donor': str(LOCAL_DONOR), 'warm_state_dir': str(STATE_DIR),
    'local_output': str(LOCAL_OUTPUT), 'drive_output': str(OUTPUT_DRIVE)
}
(LOCAL_OUTPUT/'resolved_recipe.json').write_text(json.dumps(recipe, indent=2))
print(json.dumps(recipe, indent=2))


### One canonical strict run

The launcher sets `alpha=0`, decimation 1, T=1, top-k 512, CE weight 1.0, LR 5e-4,
and gradient checkpointing wherever the exact bundled runner exposes those controls.
It captures the complete log and then applies the warm rollback gate.

Set the input/output variables below to the names used by the bundled runner. The
notebook deliberately keeps this mapping explicit rather than guessing silently.


In [ ]:
# Map paths for the exact bundled runner. These names are also recorded in effective_env.json.
import os
os.environ.update({
    'DONOR_DIR': str(LOCAL_DONOR),
    'MODEL_DIR': str(LOCAL_DONOR),
    'STATE_DIR': str(STATE_DIR),
    'WARM_STATE': str(WARM_STATE),
    'OUT_DIR': str(LOCAL_OUTPUT),
    'OUTPUT_DIR': str(LOCAL_OUTPUT),
    'TOPK': str(profile.cache_topk),
    'TEMPERATURE': '1.0',
    'CE_WEIGHT': '1.0',
    'LR': '0.0005',
    'DECIMATION': '1',
    'ALPHA_START': '0',
    'ALPHA_END': '0',
    'MN_STRICT_DEPLOY': '1',
})


In [ ]:
# Hard pre-run assertions: old unsafe recipe must be impossible.
assert os.environ['MN_STRICT_DEPLOY'] == '1'
assert float(os.environ['ALPHA_START']) == 0.0 and float(os.environ['ALPHA_END']) == 0.0
assert int(os.environ['DECIMATION']) == 1
assert float(os.environ['TEMPERATURE']) == 1.0
assert float(os.environ['CE_WEIGHT']) >= 1.0
assert int(os.environ['TOPK']) >= 512
print('Strict recipe assertions passed')


In [ ]:
# Start strict KD with the conservative legacy-cache contract.
# Change cache-mode to exact_v2 only after validate_kd_cache_v2.py succeeds.
import subprocess, sys
cmd = [
    sys.executable, str(PROJECT/'scripts/strict_kd_launcher.py'),
    '--runner', str(PROJECT/'scripts/kd_cached_recovery.py'),
    '--metrics', str(LOCAL_OUTPUT/'metrics.json'),
    '--output-dir', str(LOCAL_OUTPUT),
    '--cache-mode', 'legacy_sparse',
    '--lr', '0.0005', '--topk', str(max(512, profile.cache_topk)),
    '--temperature', '1.0', '--ce-weight', '1.0',
    '--min-improvement', '0.002', '--max-domain-regression', '0.05',
]
print(' '.join(cmd))
proc = subprocess.run(cmd, check=False)
print('return code:', proc.returncode)
if proc.returncode != 0:
    print((LOCAL_OUTPUT/'strict_kd.log').read_text(errors='replace')[-12000:])
    raise RuntimeError('strict KD failed; log preserved in LOCAL_OUTPUT')


In [ ]:
# Inspect authoritative selection. KD never replaces warm unless all gates pass.
import json
selection = json.loads((LOCAL_OUTPUT/'selection.json').read_text())
print(json.dumps(selection, indent=2, ensure_ascii=False))
assert selection['selected']['metric'] <= selection['warm']['metric'] + 1e-12


In [ ]:
# Atomic sync of logs/metadata and only the selected artifact pointer to Drive.
import os, shutil, subprocess, json, time
stage = OUTPUT_DRIVE.with_name(OUTPUT_DRIVE.name + '.staging')
if stage.exists(): shutil.rmtree(stage)
stage.mkdir(parents=True)
for name in ['preflight.json','resolved_recipe.json','effective_env.json','run_meta.json','selection.json','strict_kd.log','USE_WARM_STATE.txt','KD_ACCEPTED.txt']:
    p = LOCAL_OUTPUT/name
    if p.exists(): shutil.copy2(p, stage/name)
(stage/'SOURCE_POINTERS.json').write_text(json.dumps({
    'release_zip': str(RELEASE_ZIP), 'donor': str(DONOR_DIR), 'warm_state': str(WARM_STATE)
}, indent=2))
# Preserve an existing output as backup, then atomically rename staging.
backup = OUTPUT_DRIVE.with_name(OUTPUT_DRIVE.name + '.previous')
if backup.exists(): shutil.rmtree(backup)
if OUTPUT_DRIVE.exists(): OUTPUT_DRIVE.rename(backup)
stage.rename(OUTPUT_DRIVE)
print('Synced:', OUTPUT_DRIVE)


## Acceptance interpretation

- `KD_ACCEPTED.txt`: the candidate improved aggregate metric and stayed within every
  domain regression limit under strict `alpha=0` evaluation.
- `USE_WARM_STATE.txt`: the KD run is diagnostic only; keep `state_max.zip`.
- `selection.json` is authoritative. Never deploy `best_kd.pt` merely because that
  filename exists.
